# Data Download and Preprocessing

In this notebook, we will download the data and contextual layers needed for this project. We will also preprocess the data to make it ready for analysis.

## Table of Contents
### [Data Download](#data-download)
- **[Country Borders – UN](#countries)**
- **[Terrestrial Ecoregions of the World](#ecoregions)**
- **[Protected Areas - WDPA](#protected_areas)**
- **[MODIS/Terra Net Primary Production](#npp)**
- **[Anthropogenic Biomes of the World](#anthropogenic)**
- **[World Dryland Areas](#dryland)**
- **[Global Livestock Production Systems](#glps)**

## Setup

### Library import


In [ ]:
import glob
import os
from netrc import netrc

import geopandas as gpd
import lonboard as lb
import rasterio
from lonboard.colormap import apply_categorical_cmap
from osgeo import gdal
from rasterio.warp import Resampling, calculate_default_transform, reproject
from rio_cogeo.cogeo import cog_translate
from rio_cogeo.profiles import cog_profiles

from src.data_download import (
    AnthropogenicBioms,
    HDFProcessor,
    download_and_unzip,
    get_links,
    unzip_file,
)

<a id='data-download'></a>
## Data Download

We will download the data from the sources listed on this spreadsheet: [Tentative list of Datasets](https://docs.google.com/spreadsheets/d/1fkxozbhOsqNVV0pNf_1lTOv6UaA-yzyMh4bzURbmlrg/edit?usp=sharing)

<a id='countries'></a>
### Country Borders – UN ([source](https://ec.europa.eu/eurostat/web/gisco/geodata/reference-data/administrative-units-statistical-units/countries#countries20))

**Download data**

In [ ]:
url = "https://gisco-services.ec.europa.eu/distribution/v2/countries/download/ref-countries-2020-01m.shp.zip"
directory = "../data/raw/"
download_and_unzip(url, directory)

In [ ]:
zip_file_path = "../data/raw/ref-countries-2020-01m.shp/CNTR_RG_01M_2020_4326.shp.zip"
unzip_file(zip_file_path)

**Save as `parquet`**

In [ ]:
data_path = (
    "../data/raw/ref-countries-2020-01m.shp/CNTR_RG_01M_2020_4326.shp/CNTR_RG_01M_2020_4326.shp"
)
output_path = "../data/processed/countries.parquet"
gdf = gpd.read_file(data_path)
gdf.to_parquet(output_path, index=False)

**Save as `GeoJSON`**

In [ ]:
gdf.to_file("../data/processed/countries.geojson", driver="GeoJSON")

**Read data**

In [ ]:
data_path = "../data/processed/countries.parquet"
countries = gpd.read_parquet(data_path)
countries.head()

**Display data on map**

In [ ]:
polys = lb.SolidPolygonLayer.from_geopandas(
    countries,
    wireframe=True,
    get_fill_color=[6, 204, 204, 30],
    get_line_color=[6, 204, 204, 255],
    extruded=True,
)
m = lb.Map(polys)
m

<a id='ecoregions'></a>
### Terrestrial Ecoregions of the World ([source](https://www.worldwildlife.org/publications/terrestrial-ecoregions-of-the-world))

**Citation** 

Olson, D. M., Dinerstein, E., Wikramanayake, E. D., Burgess, N. D., Powell, G. V. N., Underwood, E. C., D'Amico, J. A., Itoua, I., Strand, H. E., Morrison, J. C., Loucks, C. J., Allnutt, T. F., Ricketts, T. H., Kura, Y., Lamoreux, J. F., Wettengel, W. W., Hedao, P., Kassem, K. R. 2001. Terrestrial ecoregions of the world: a new map of life on Earth. Bioscience 51(11):933-938.

**Not all the ecoregions are applicable in this assignment**. Only those related to rangelands. According to the rangelands atlas (based on the previous version of the map from 2001): 
- Deserts and xeric shrublands 
- Flooded grasslands and savannas 
- Mediterranean forests, woodlands, and scrub 
- Montane grasslands and shrublands 
- Temperate grasslands, savannas, and shrublands 
- Tropical and subtropical grasslands, savannas, and shrublands 
- Tundra


**Download data**

In [ ]:
url = "https://storage.googleapis.com/teow2016/Ecoregions2017.zip"
directory = "../data/raw/"
download_and_unzip(url, directory)

**Read data**

In [ ]:
url = "https://storage.googleapis.com/teow2016/Ecoregions2017.zip"
output_path = "../data/processed/ecoregions_2017.parquet"
gdf = gpd.read_file(url, engine="pyogrio")

**Save as `parquet`**

In [ ]:
gdf.to_parquet(output_path, index=False)

**Read data from `parquet`**

In [ ]:
data_path = "../data/processed/ecoregions_2017.parquet"
ecoregions = gpd.read_parquet(data_path)
ecoregions.head()

**Filter data**

In [ ]:
rangeland_ecoregions = [
    "Tundra",
    "Mediterranean Forests, Woodlands & Scrub",
    "Deserts & Xeric Shrublands",
    "Temperate Grasslands, Savannas & Shrublands",
    "Montane Grasslands & Shrublands",
    "Flooded Grasslands & Savannas",
    "Tropical & Subtropical Grasslands, Savannas & Shrublands",
]

In [ ]:
ecoregions = ecoregions[ecoregions["BIOME_NAME"].isin(rangeland_ecoregions)]

**Display data on map**

In [ ]:
color_map = {
    "Tundra": [181, 197, 143],
    "Mediterranean Forests, Woodlands & Scrub": [204, 184, 121],
    "Deserts & Xeric Shrublands": [223, 223, 194],
    "Temperate Grasslands, Savannas & Shrublands": [220, 217, 57],
    "Montane Grasslands & Shrublands": [171, 108, 40],
    "Flooded Grasslands & Savannas": [184, 217, 235],
    "Tropical & Subtropical Grasslands, Savannas & Shrublands": [108, 159, 184],
}

In [ ]:
ecoregions_layer = lb.SolidPolygonLayer.from_geopandas(ecoregions)

ecoregions_layer.get_fill_color = apply_categorical_cmap(
    values=ecoregions["BIOME_NAME"].astype("category"), cmap=color_map, alpha=150
)

m = lb.Map(ecoregions_layer)
m

**Save as `GeoJSON`**

In [ ]:
ecoregions.to_file("../data/processed/ecoregions_2017.geojson", driver="GeoJSON")

<a id='protected_areas'></a>
### Protected Areas - WDPA ([source](https://www.protectedplanet.net/en/thematic-areas/wdpa?tab=WDPA))

**Citation**

UNEP-WCMC and IUCN (2024), Protected Planet: The World Database on Protected Areas (WDPA) [Online], April 2024, Cambridge, UK: UNEP-WCMC and IUCN. Available at: [www.protectedplanet.net]([www.protectedplanet.net).

**Download data**

In [ ]:
url = "https://d1gam3xoknrgr2.cloudfront.net/current/WDPA_Apr2024_Public_shp.zip"
directory = "../data/raw/"
download_and_unzip(url, directory)

In [ ]:
directory = "../data/raw/WDPA_Apr2024_Public_shp"
zip_files = [
    "WDPA_Apr2024_Public_shp_0.zip",
    "WDPA_Apr2024_Public_shp_1.zip",
    "WDPA_Apr2024_Public_shp_2.zip",
]
for zip_file in zip_files:
    zip_file_path = os.path.join(directory, zip_file)
    unzip_file(zip_file_path)

**[Preprocess data](02_WDPA_processing.ipynb)**

**Read data from `parquet`**

In [ ]:
data_path = "../data/processed/wdpa.parquet"
wdpa = gpd.read_parquet(data_path)
wdpa.head()

**Display data on map**

In [ ]:
wdpa["IUCN_CAT"].unique()

In [ ]:
color_map = {
    "Ia": [255, 105, 180],  # Hot Pink
    "Ib": [255, 140, 0],  # Dark Orange
    "II": [0, 255, 0],  # Lime
    "III": [0, 0, 255],  # Blue
    "IV": [255, 0, 255],  # Magenta
    "V": [0, 255, 255],  # Aqua
    "VI": [255, 255, 0],  # Yellow
    "Not Applicable": [128, 128, 128],  # Gray
    "Not Assigned": [128, 128, 128],  # Gray
    "Not Reported": [128, 128, 128],  # Gray
}

In [ ]:
wdpa_layer = lb.SolidPolygonLayer.from_geopandas(wdpa)

wdpa_layer.get_fill_color = apply_categorical_cmap(
    values=wdpa["IUCN_CAT"].astype("category"), cmap=color_map, alpha=150
)

m = lb.Map(wdpa_layer)
m

<a id='npp'></a>
### MODIS/Terra Net Primary Production ([source](https://lpdaac.usgs.gov/products/mod17a3hgfv061/))

**Description**

The MOD17A3HGF Version 6.1 product provides information about annual Gross and Net Primary Production (GPP and NPP) at 500 meter (m) pixel resolution. Annual Terra Moderate Resolution Imaging Spectroradiometer (MODIS) GPP and NPP is derived from the sum of all 8-day GPP Net Photosynthesis (PSN) products (MOD17A2H) from the given year. The PSN value is the difference of the GPP and the Maintenance Respiration (MR).

The MOD17A3HGF will be generated at the end of each year when the entire yearly 8-day MOD15A2H is available. Hence, the gap-filled MOD17A3HGF is the improved MOD17, which has cleaned the poor-quality inputs from 8-day Leaf Area Index and Fraction of Photosynthetically Active Radiation (LAI/FPAR) based on the Quality Control (QC) label for every pixel. If any LAI/FPAR pixel did not meet the quality screening criteria, its value is determined through linear interpolation. However, users cannot get MOD17A3HGF in near-real time because it will be generated only at the end of a given year.

**Citation** 

Running, S., M. Zhao.  *MODIS/Terra Net Primary Production Gap-Filled Yearly L4 Global 500m SIN Grid V061*. 2021, distributed by NASA EOSDIS Land Processes Distributed Active Archive Center, https://doi.org/10.5067/MODIS/MOD17A3HGF.061. Accessed 2024-04-19.

**NASA Earthdata Login**

You will need a NASA Earthdata Login account in order to download LP DAAC data (and consequently use this script). To create a NASA Earthdata Login account, go to the [Earthdata Login website](https://urs.earthdata.nasa.gov/) and click the “Register” button, which is next to the green “Log In” button under the Password entry box. Fill in the required boxes (indicated with a red asterisk), then click on the “Register for Earthdata Login” green button at the bottom of the page. An email with instructions for activating the registration completes the process.

To download data from the LP DAAC archive, you need to authorize our applications to view your NASA Earthdata Login profile. Once authorization is complete, you may resume your session. To authorize Data Pool, please [click here](https://urs.earthdata.nasa.gov/approve_app?client_id=ijpRZvb9qeKCK5ctsn75Tg&_ga=2.128429068.1284688367.1541426539-1515316899.1516123516).

In [ ]:
# authentication url
url = "urs.earthdata.nasa.gov"

In [ ]:
# make the netrc directory
netrc_folder = os.path.expanduser("~/.netrc")

In [ ]:
# Get user name and password
usr = netrc(netrc_folder).authenticators(url)[0]
pwd = netrc(netrc_folder).authenticators(url)[2]

#### Create a layer for each year

We will download the data from the [Data Pool](https://lpdaac.usgs.gov/tools/data-pool/):

The Data Pool is the publicly available portion of the LP DAAC online holdings. Data Pool provides a direct way to access data product files via HTTPS. All Data Pool holdings are available at no cost.

The satellite data is composed of small `.hdf` files that contain part of the global map. We need to combine these into a single map (per year of data) for further analysis e.g. converting to Cloud Optimized GeoTIFF (COG) format.

In [ ]:
base_url = "https://e4ftl01.cr.usgs.gov/MOLT/MOD17A3HGF.061"
base_path = "../data/raw/MOD17A3HGF"
subdataset_name = "SUBDATASET_2_NAME"
dataset_name = "MOD17A3HGF_NPP"
os.makedirs(base_path, exist_ok=True)

In [ ]:
# Get the year directories
links = get_links(base_url)
folder_links = [link.get("href") for link in links if link.get("href").endswith("01/")]

In [ ]:
processor = HDFProcessor(base_url, base_path, usr, pwd, subdataset_name, dataset_name)
for folder in folder_links[19:20]:
    processor.convert_hdf_files_to_cog(folder)

**Set `no-data` value**

In [ ]:
# Get the list of geotif files
base_path = "../data/raw/MOD17A3HGF"
geotif_files = sorted(glob.glob(f"{base_path}/*.tif"))

# Set a threshold value
threshold = 32760  # replace with your threshold

for input_file in geotif_files:
    file_name = os.path.basename(input_file)
    new_file_name = file_name.replace("_cog_", "_cog_nodata_")
    output_file = os.path.join(base_path, new_file_name)

    # Open the existing GeoTIFF file
    with rasterio.open(input_file) as src:
        # Read the data into a numpy array
        data = src.read(1)

        # Replace all values above the threshold with -1
        data[data > threshold] = -1

        # Define the COG conversion configuration
        config = cog_profiles.get("deflate")
        config["nodata"] = -1

        # Write the updated data back to a new GeoTIFF file
        with rasterio.open(output_file, "w", **src.meta) as dst:
            dst.write(data, 1)

        # Convert the GeoTIFF file to a COG and set the NoData value
        cog_translate(output_file, output_file, config)

# Remove the original GeoTIFF files
for input_file in geotif_files:
    os.remove(input_file)

In [ ]:
# Get the list of geotif files
base_path = "../data/raw/MOD17A3HGF"
output_path = "../data/processed/MOD17A3HGF"
geotif_files = sorted(glob.glob(f"{base_path}/*.tif"))

# Set a threshold value
threshold = 32760  # replace with your threshold

# Define the new CRS
dst_crs = "EPSG:4326"  # replace with your desired CRS

for input_file in geotif_files[1:]:
    file_name = os.path.basename(input_file)
    new_file_name = file_name.replace("_cog_nodata_", "_cog_nodata_reproj_")
    output_file = os.path.join(output_path, new_file_name)

    # Open the existing GeoTIFF file
    with rasterio.open(input_file) as src:
        # Read the data into a numpy array
        data = src.read(1)

        # Replace all values above the threshold with -1
        data[data > threshold] = -1

        # Define the COG conversion configuration
        config = cog_profiles.get("deflate")
        config["nodata"] = -1

        # Calculate the ideal dimensions and transformation in the new crs
        transform, width, height = calculate_default_transform(
            src.crs, dst_crs, src.width, src.height, *src.bounds
        )

        # Update the metadata
        kwargs = src.meta.copy()
        kwargs.update({"crs": dst_crs, "transform": transform, "width": width, "height": height})

        # Reproject and write the reprojected data to a new GeoTIFF file
        with rasterio.open(output_file, "w", **kwargs) as dst:
            reproject(
                source=rasterio.band(src, 1),
                destination=rasterio.band(dst, 1),
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=transform,
                dst_crs=dst_crs,
                resampling=Resampling.nearest,
            )

        # Convert the GeoTIFF file to a COG and set the NoData value
        cog_translate(output_file, output_file, config)

## Remove the original GeoTIFF files
# for input_file in geotif_files:
#    os.remove(input_file)

<a id='anthropogenic'></a>
### Anthropogenic Biomes of the World, v2 (2000) ([source](https://sedac.ciesin.columbia.edu/data/set/anthromes-anthropogenic-biomes-world-v2-2000/data-download))

**Citation:**

Ellis, E.C., K.K. Goldewijk, S. Siebert, D. Lightman, and N. Ramankutty. 2010. Anthropogenic Transformation of the Biomes, 1700 to 2000. Global Ecology and Biogeography 19 (5): 589-606. https://doi.org/10.1111/j.1466-8238.2010.00540.x.

**Notes:**

This dataset is hosted by NASA so you need to configure a connection to download data from an Earthdata Login enabled server. The code used here can be found in [How To Access Data With Python](https://urs.earthdata.nasa.gov/documentation/for_users/data_access/python) provides more information. Note that you will need to a secure way to configure the Earthdata Login username and password.

To create a NASA Earthdata Login account, go to the [Earthdata Login website](https://urs.earthdata.nasa.gov/) and click the “Register” button, which is next to the green “Log In” button under the Password entry box. Fill in the required boxes (indicated with a red asterisk), then click on the “Register for Earthdata Login” green button at the bottom of the page. An email with instructions for activating the registration completes the process.

In [ ]:
# Get the Earth Data username and password
env_path = os.path.abspath(os.path.join(os.getcwd(), os.pardir, ".env"))
env = {}


with open(env_path, "r") as file:
    for line in file:
        line = line.strip()
        if line:
            key, value = line.split("=", 1)
            env[key] = value

username = env["earth_data_username"]
password = env["earth_data_key"]

In [ ]:
# Define the URL and target path
url = "https://sedac.ciesin.columbia.edu/downloads/data/anthromes/anthromes-anthropogenic-biomes-world-v2-2000/anthromes-v2-2000-global-geotif.zip"
target_path = "../data/raw"

In [ ]:
# Download the file
session = AnthropogenicBioms(username, password)

os.makedirs(target_path, exist_ok=True)  # Ensure the target directory exists

zip_file_path = session.download_file(url, target_path)  # Download the file

session.unzip_file(
    zip_file_path, target_path
)  # Unzip the file in a folder within the same directory where it was downloaded

<a id='dryland'></a>
### World Dryland Areas ([source](https://datadownload.unep-wcmc.org/requests/new?dataset=Drylands_dataset_2007))

**Description**

The original drylands dataset was developed to define the Millennium Ecosystem Assessment (MA)
dryland system boundary. Sörensen (2007) used this map to further delineate dryland areas of
relevance to the CBD Programme of Work on Dry and Subhumid Lands. This current dataset builds
on these maps to incorporate additional features to update the data to better reflect dry and sub-
humid tropical forests in accordance with paragraph 13 of Convention on Biological Diversity (CBD)
CoP Decision IX/17.

**Citation** 

UNEP-WCMC, 2007 A spatial analysis approach to the global delineation of dryland
areas of relevance to the CBD Programme of Work on Dry and Subhumid Lands. Dataset based on
spatial analysis between WWF terrestrial ecoregions (WWF-US, 2004) and aridity zones (CRU/UEA;
UNEPGRID, 1991). Dataset checked and refined to remove many gaps, overlaps and slivers (July
2014).

**Read data**

In [ ]:
data_path = (
    "../data/raw/Drylands_dataset_2007/Drylands_latest_July2014/drylands_UNCCD_CBD_july2014.shp"
)
drylands = gpd.read_file(data_path)
drylands["UNCCDDESC"] = drylands["UNCCDDESC"].fillna("additional areas")
drylands.head()

**Display data on map**

In [ ]:
color_map = {
    "arid: P/PET 0.05 - 0.20": [251, 212, 130],
    "semiarid: P/PET 0.20 - 0.50": [250, 171, 35],
    "dry subhumid: P/PET 0.50 - 0.65": [224, 0, 18],
    "additional areas": [251, 255, 48],
}

In [ ]:
drylands_layer = lb.SolidPolygonLayer.from_geopandas(drylands)

drylands_layer.get_fill_color = apply_categorical_cmap(
    values=drylands["UNCCDDESC"].astype("category"), cmap=color_map, alpha=150
)

m = lb.Map(drylands_layer)
m

**Save as `GeoJSON`**

In [ ]:
drylands.to_file("../data/processed/drylands_2007.geojson", driver="GeoJSON")

<a id='glps'></a>
### Global Livestock Production Systems, v.5 ([source](https://datadownload.unep-wcmc.org/requests/new?dataset=Drylands_dataset_2007))

**Description**

The global livestock sector is rapidly changing in response to globalization and growing demand for animal-source foods, driven by population growth and increasing wealth in much of the developing world.
As well as the many benefits and opportunities associated with rapid sector transformation and growth, they are also associated with social, environmental and public health risks. There are huge differences in the ways in which livestock are kept in different places and what their roles are. Hence, we need to develop a good understanding of the differences among production systems if we are to be able to help poor livestock keepers take advantage of the rising demand for animal-source foods, help livestock keepers adapt to a changing and more volatile climate; minimize the risk of disease emergence and spread, not only among livestock but also in people; and to help all livestock keepers mitigate greenhouse gas emissions via a wide range of options.
The mapping of the Global Livestock Production System is the result of a a long-standing collaboration between FAO and the International Livestock Research Institute (ILRI). The first attempt to map livestock production systems, at least in the developing world, was by Thornton et al. in 2002, based on a classification scheme developed by Seré and Steinfeld in 1996. This version (2011) includes more accurate and higher spatial resolution (circa 1 km) input data and updates the FAO-ILRI previous version. Main reference for this 2011 map is the forthcoming publication of Robinson et al. (2011). This new book provides references to the most up to date map of global livestock production systems and revised estimates of the number of poor livestock keepers, globally, within the different production systems. It proposes alternative approaches to mapping production systems that are explicitly linked to livelihoods, and reviews the ways in which intensive production can be accounted for. The book also underscores the areas that need further development. The FAO and ILRI continue to work jointly on several of these.

**Download data**

In [ ]:
url = "https://storage.googleapis.com/fao-maps-catalog-data/geonetwork/livestock/GLPS_2011.zip"
directory = "../data/raw/"
download_and_unzip(url, directory)

**Create COG**

In [ ]:
# Convert the ESRI Grid file to a COG
input_file = "../data/raw/GLPS_2011/ps_cmb"
output_file = "../data/processed/GLPS_cog_2011.tif"

gdal.Translate(
    output_file, input_file, format="COG", creationOptions=["COMPRESS=DEFLATE"], noData=0
)

In [ ]:
# Define the input and output file paths
input_file = "../data/raw/GLPS_2011/ps_cmb.ovr"
output_file = "../data/processed/GLPS_2011.tif"

# Define the profile for the output file
output_profile = cog_profiles.get("deflate")

# Transform the .ovr file to a COG
cog_translate(input_file, output_file, output_profile)